# Laboratorio 2.2 · Historia, gold y calidad de datos

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la tienda. En la 2.1 dejamos los pedidos limpios, sin duplicados y en pesos.
Hoy les damos **contexto con historia**, construimos gold y le ponemos **un portero** a silver.

| Parte | Qué haces | La pregunta |
|---|---|---|
| 1 | Las fotos del CRM | ¿Qué cambió entre una foto y la siguiente? |
| 2 | SCD tipo 1 | ¿En qué ciudad vive el cliente hoy? |
| 3 | SCD tipo 2 | ¿En qué ciudad vivía cuando compró? |
| 4 | Join por punto en el tiempo | ¿Cuánto cambia el reporte de ventas por ciudad? |
| 5 | Gold | La tabla que ve el tablero |
| 6 | Contrato con Pandera | ¿Qué entra a silver y qué no? |
| 7 | Un lote roto, del ruidoso | ¿Se detiene el pipeline? |
| 8 | Un lote roto, del silencioso | ¿Y si cada fila, una por una, se ve bien? |
| 9 | El pipeline completo | Idempotencia y falla cerrada |

## Preparación

In [ ]:
%%capture
!pip -q install -U "polars>=1.10" "pandera[polars]>=0.20" pyarrow pytest

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(".")
print("Paquete transformacion instalado")

In [ ]:
import inspect, warnings
from datetime import date
import polars as pl
from IPython.display import Code, display
from transformacion import generar_bronze, leer, limpieza, dedup, enriquecer, scd, calidad, gold, pipeline

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config(tbl_rows=16, tbl_cols=12, fmt_str_lengths=40, tbl_width_chars=160)

def ver(funcion):
    display(Code(inspect.getsource(funcion), language="python"))

LANDING = "landing"
print(generar_bronze.generar(LANDING))     # 14 cargas: del 08/09 al 21/09/2026
b = leer.todas(LANDING)                      # un DataFrame por tabla de bronze
print("Polars", pl.__version__)

In [ ]:
# Lo que construimos en la 2.1, en tres líneas
tc = enriquecer.tipo_cambio_diario(b["tipo_cambio"])
unicos = dedup.ultima_version(limpieza.pedidos(b["pedidos_web"], b["pedidos_tienda"]))
pedidos = enriquecer.con_tipo_cambio(unicos.filter(pl.col("total").is_not_null()), tc)
print(f"{len(pedidos):,} pedidos en pesos")

## Parte 1 · Las fotos del CRM

El CRM no tiene CDC ni marca de actualización: cada noche manda **la foto completa** de todos sus clientes.
Catorce fotos, una por carga. El cambio no viene marcado: hay que **descubrirlo** comparando una foto con la anterior.

In [ ]:
fotos = limpieza.clientes(b["clientes"])          # normalizadas: ciudad canónica, segmento y correo en minúsculas
fotos.group_by("fecha_carga").agg(clientes=pl.len()).sort("fecha_carga")

In [ ]:
CLIENTE = 42
fotos.filter(pl.col("cliente_id") == CLIENTE).select("fecha_carga", "ciudad", "segmento", "email").sort("fecha_carga")

El cliente 42 se mudó a Tijuana... y tres días después regresó a CDMX. ¿Cómo lo guardamos?

## Parte 2 · SCD tipo 1: sobrescribir

La dimensión dice cómo es el cliente **hoy**. Fácil, barata y sin historia.

In [ ]:
ver(scd.scd1)

In [ ]:
dim1 = None
for (fecha,), foto in fotos.sort("fecha_carga").group_by("fecha_carga", maintain_order=True):
    dim1 = scd.scd1(dim1, foto)
print(f"Clientes: {len(dim1):,}   ·   activos: {dim1['activo'].sum():,}   ·   bajas: {(~dim1['activo']).sum()}")
dim1.filter(pl.col("cliente_id") == CLIENTE)

Para el cliente 42 la mudanza **nunca existió**. Las bajas no se borran: se marcan inactivas, porque hay pedidos que las referencian.

## Parte 3 · SCD tipo 2: versionar

Cada cambio **cierra** la versión vigente (`valido_hasta`) y **abre** una nueva (`valido_desde`).
La dimensión es híbrida, como casi siempre en la vida real: ciudad y segmento son tipo 2; el correo es tipo 1.

In [ ]:
ver(scd.scd2)

Apliquemos las fotos una por una y miremos al cliente 42:

In [ ]:
orden = fotos.sort("fecha_carga").partition_by("fecha_carga", maintain_order=True, as_dict=True)
historia = None
for (fecha,), foto in orden.items():
    historia = scd.scd2(historia, foto, fecha)
    if fecha in (date(2026, 9, 8), date(2026, 9, 12), date(2026, 9, 15)):
        print(f"--- después de la foto del {fecha}")
        display(scd.historia_de(historia, CLIENTE).drop("email"))

- La primera versión vale **desde siempre** (`1900-01-01`): si no, los pedidos de junio no encontrarían a su cliente.
- La versión vigente vale **hasta nuevo aviso** (`9999-12-31`): así una sola comparación sirve para todas las filas.
- La precisión es **de un día**: la foto del 12 dice que el cambio ocurrió en algún momento del 11.

### Dos propiedades que no son negociables

In [ ]:
ultima_fecha, ultima_foto = list(orden.items())[-1]
otra_vez = scd.scd2(historia, ultima_foto, ultima_fecha[0])
print("1) Aplicar dos veces la misma foto no cambia nada:", otra_vez.equals(historia))

try:
    scd.scd2(historia, orden[(date(2026, 9, 10),)], date(2026, 9, 10))
except scd.FotoFueraDeOrden as e:
    print("2) Una foto vieja después de una nueva se rechaza:\n  ", e)

Si se reprocesa un día viejo, la salida honesta es **reconstruir** la dimensión desde todas las fotos, en orden.
Con 14 fotos tarda milisegundos; con 5 años de fotos diarias ya es una decisión de diseño.

In [ ]:
ver(scd.reconstruir)
historia = scd.reconstruir(fotos)
print(f"Versiones: {len(historia):,}   ·   vigentes: {historia['vigente'].sum():,}   ·   clientes con historia: "
      f"{historia.filter(pl.col('version') > 1)['cliente_id'].n_unique():,}")

### La trampa: historizar antes de limpiar

¿Qué pasa si construimos la SCD2 con las fotos **crudas**, sin normalizar la ciudad?

In [ ]:
crudas = b["clientes"].select("cliente_id", "ciudad", "segmento", "email", "fecha_carga")
historia_cruda = scd.reconstruir(crudas)
print(f"Versiones con fotos crudas: {len(historia_cruda):,}   contra   {len(historia):,} con fotos limpias")
falsos = historia_cruda.filter(pl.col("version") > 1).join(historia.filter(pl.col("version") > 1), on=["cliente_id", "valido_desde"], how="anti")
scd.historia_de(historia_cruda, falsos["cliente_id"][0]).select("cliente_id", "ciudad", "segmento", "valido_desde", "valido_hasta")

Miles de «mudanzas» que no existen: `CDMX` → `cdmx` → `Ciudad de México`. **Se normaliza antes de historizar.**

## Parte 4 · Join por punto en el tiempo

Cada pedido se une con la versión del cliente **vigente el día de la venta**: `valido_desde <= fecha_negocio < valido_hasta`.

In [ ]:
ver(enriquecer.con_cliente_historico)

In [ ]:
actual = enriquecer.con_cliente_actual(pedidos, historia.filter("vigente"))
historico = enriquecer.con_cliente_historico(pedidos, historia)
desde = date(2026, 9, 8)          # desde la primera foto, cuando empiezan a registrarse los cambios
comparacion = (gold.ventas_por_ciudad(actual, desde).rename({"pedidos": "pedidos_tipo1", "ingreso_mxn": "ingreso_tipo1"})
               .join(gold.ventas_por_ciudad(historico, desde).rename({"pedidos": "pedidos_tipo2", "ingreso_mxn": "ingreso_tipo2"}), on="ciudad")
               .with_columns(diferencia=(pl.col("ingreso_tipo1") - pl.col("ingreso_tipo2")).round(0))
               .sort(pl.col("diferencia").abs(), descending=True))
comparacion

Con tipo 1, las ventas **se mudan con el cliente**: si alguien que compraba en Guadalajara se fue a Puebla,
todas sus compras viejas aparecen en Puebla. Ninguna de las dos es «la correcta» en abstracto:

| Pregunta de negocio | Dimensión |
|---|---|
| ¿Dónde están hoy mis clientes vip? (campaña) | Tipo 1 |
| ¿Cuánto vendió Guadalajara en septiembre? (finanzas) | Tipo 2 |

### ¿Y la ciudad «Desconocida»?

In [ ]:
desconocidos = historico.filter(pl.col("ciudad") == "Desconocida")
pl.DataFrame({"causa": ["Venta de mostrador sin cliente (cliente_id = -1)", "El cliente no existe en el CRM"],
              "pedidos": [len(desconocidos.filter(pl.col("cliente_id") == limpieza.CLIENTE_DESCONOCIDO)),
                          len(enriquecer.sin_cliente(historico))]})

Las primeras son legítimas (el **miembro desconocido** del modelo dimensional). Las segundas son un problema de la fuente:
en un pipeline serio se reportan al dueño del dato, no se esconden.

## Parte 5 · Gold

Gold solo **agrega**. Toda la lógica de limpieza ya ocurrió en silver: si en gold aparece un `CASE WHEN` que corrige datos,
está en la capa equivocada.

In [ ]:
ver(gold.ventas_diarias)
ventas = gold.ventas_diarias(historico)
(ventas.filter(pl.col("fecha_negocio") >= date(2026, 9, 14))
       .group_by("fecha_negocio").agg(pl.col("pedidos").sum(), pl.col("ingreso_mxn").sum().round(0)).sort("fecha_negocio"))

¿Y el 21? No es que ese día no se vendiera: la última carga es de las 3 de la madrugada, así que el día **va empezando**.
El último día de un reporte casi siempre está incompleto; el tablero tiene que decirlo (o no mostrarlo) para que nadie lo lea como una caída.

## Parte 6 · El contrato de silver con Pandera

Hasta aquí filtramos los totales nulos «a mano». Un pipeline de verdad tiene un **contrato** explícito:
qué columnas, qué tipos y qué reglas debe cumplir cada fila para entrar a silver.

In [ ]:
ver(calidad.PedidoSilver)

In [ ]:
validas, cuarentena = calidad.validar(unicos)
print(f"Válidas: {len(validas):,}   ·   en cuarentena: {len(cuarentena):,}")
calidad.por_regla(cuarentena)

In [ ]:
ver(calidad.validar)

Tres niveles de severidad, tres reacciones distintas:

| Nivel | Ejemplo | Reacción |
|---|---|---|
| Esquema | Falta la columna `moneda` | Se detiene todo: no se puede confiar en nada |
| Fila | Un total nulo | Esa fila va a cuarentena, con su motivo; el resto sigue |
| Lote | 30 % de una carga en cuarentena | Se detiene la carga: algo cambió en el origen |

In [ ]:
try:
    calidad.validar(unicos.drop("moneda"))
except calidad.ErrorDeEsquema as e:
    print(e)

In [ ]:
calidad.rechazo_por_carga(validas, cuarentena).tail(5)

## Parte 7 · Un lote roto, del ruidoso

El 22/09 el sistema de la tienda en línea se actualizó. Nadie avisó. Generemos esa carga:

In [ ]:
generar_bronze.generar(LANDING, lote_malo="ruidoso")
b = leer.todas(LANDING)
unicos = dedup.ultima_version(limpieza.pedidos(b["pedidos_web"], b["pedidos_tienda"]))
validas, cuarentena = calidad.validar(unicos)
calidad.por_regla(cuarentena.filter(pl.col("fecha_carga") == date(2026, 9, 22)))

Tres cosas a la vez: totales gigantes (el origen empezó a mandar **centavos**), un estado nuevo (`pagado_pendiente`) y monedas nulas.
¿Cuánto pesa esto?

In [ ]:
por_carga = calidad.rechazo_por_carga(validas, cuarentena)
total = por_carga.select(pl.col("cuarentena").sum() / pl.col("filas").sum()).item()
print(f"Rechazo contra TODA la historia: {total:.1%}  <- con un umbral de 2 %, pasaría")
por_carga.tail(3)

Contra toda la historia, el lote roto **se diluye** y pasa. Por eso el umbral se mide **por carga**.
Ahora el pipeline completo:

In [ ]:
try:
    pipeline.correr(LANDING, "lakehouse")
except calidad.LoteRechazado as e:
    print(e)

## Parte 8 · Un lote roto, del silencioso

Otra versión del mismo 22/09: esta vez el sistema manda los totales **en dólares, con la etiqueta MXN**.
Cada fila, una por una, se ve perfecta: total positivo, menor al máximo, moneda válida.

In [ ]:
generar_bronze.generar(LANDING, lote_malo="silencioso")
b = leer.todas(LANDING)
unicos = dedup.ultima_version(limpieza.pedidos(b["pedidos_web"], b["pedidos_tienda"]))
validas, cuarentena = calidad.validar(unicos)
calidad.rechazo_por_carga(validas, cuarentena).tail(2)

Ninguna regla por fila lo ve. Hace falta una regla **de lote**: ¿se parece esta carga a las de siempre?

In [ ]:
ver(calidad.distribucion_por_carga)
calidad.distribucion_por_carga(validas).tail(4)

In [ ]:
try:
    pipeline.correr(LANDING, "lakehouse")
except calidad.LoteRechazado as e:
    print(e)

Las reglas por fila protegen de valores imposibles; las reglas de lote, de valores **posibles pero falsos**.
Un pipeline serio necesita las dos.

## Parte 9 · El pipeline completo

Regresamos a las 14 cargas buenas y corremos el pipeline como lo haría el orquestador: por línea de comandos y leyendo el **código de salida**.

In [ ]:
generar_bronze.generar(LANDING)
!python -m transformacion.pipeline --landing landing --salida lakehouse --metricas corrida1.json > /dev/null; echo "código de salida: $?"
!python -m transformacion.pipeline --landing landing --salida lakehouse --metricas corrida2.json > /dev/null; echo "código de salida: $?"
import json
c1, c2 = json.load(open("corrida1.json")), json.load(open("corrida2.json"))
print("huella de gold:", c1["huella_gold"], c2["huella_gold"], "-> idempotente" if c1["huella_gold"] == c2["huella_gold"] else "-> ¡NO es idempotente!")
!find lakehouse -name "*.parquet" | sort

Y la falla cerrada: con el lote ruidoso, el pipeline termina con código 2 y **gold queda como estaba**.

In [ ]:
generar_bronze.generar(LANDING, lote_malo="ruidoso")
!python -m transformacion.pipeline --landing landing --salida lakehouse > /dev/null 2> error.txt; echo "código de salida: $?"; tail -1 error.txt
print("huella de gold después del intento:", gold.huella(pl.read_parquet("lakehouse/gold/ventas_diarias.parquet")))

In [ ]:
ver(pipeline.publicar)

## Cierre

| Hoy viste | Principio |
|---|---|
| SCD tipo 1 | La foto de hoy; barata, sin historia |
| SCD tipo 2 | Versiones con rango de validez; idempotente y en orden |
| Normalizar antes de historizar | Si no, la historia se llena de cambios falsos |
| Join por punto en el tiempo | El contexto del día de la venta |
| Contrato con Pandera | Esquema, fila y lote: tres severidades |
| Umbral por carga | Contra la historia, el problema se diluye |
| Reglas de lote | Lo posible pero falso |
| Falla cerrada | Si algo falla, no se publica nada |

**Tarea:** la sección 2.1 y 2.2 del kit local (`kit_pipeline_2_2.zip`).

**Próxima sesión (2.3):** lo mismo, pero **dentro del motor**: ELT con dbt sobre DuckDB. La SCD2 que hoy escribiste a mano,
dbt la hace con un `snapshot`.